In [ ]:
# Install external packages
%pip -q install tldextract transformers tqdm joblib matplotlib


In [ ]:
# Import libraries, select GPU/CPU, and set random seed

import os, re, math, json, time, random, urllib.parse
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tldextract, joblib
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve, auc
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
DEVICE


In [ ]:
# Experiment configuration

CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 6,
    "patience": 2,
    "dropout": 0.2,
    "lr_bert": 5e-6,
    "lr_head": 5e-4,
    "weight_decay": 1e-2,
    "threshold": 0.5,
    "freeze_bert_epochs": 2,
    "sample_train": 140000,
    "sample_val": 30000,
    "sample_test": 30000,
    "use_sample": True,
    "pooling_options": ["mean"],
    "save_dir": "ablation_exp4_large_sample_140k_30k_30k",
}
os.makedirs(CONFIG["save_dir"], exist_ok=True)
CONFIG


In [ ]:
# Upload CSV dataset and read it into a pandas DataFrame
from google.colab import files
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
df = pd.read_csv(CSV_PATH)
print(df.shape)
print(df.columns.tolist()[:20])
df.head()


In [ ]:
# Normalize labels/URLs and deduplicate on a canonical URL key

import numpy as np
import pandas as pd
from urllib.parse import urlsplit, urlunsplit
import tldextract

extractor = tldextract.TLDExtract(suffix_list_urls=())

if "url" not in df.columns:
    for cand in ["URL", "Url", "uRL", "link", "Link"]:
        if cand in df.columns:
            df = df.rename(columns={cand: "url"})
            break
if "url" not in df.columns:
    raise ValueError("Rename URL column to 'url'.")

if "label" not in df.columns:
    for cand in ["label", "class", "status", "type", "result", "Category", "Label", "target", "y"]:
        if cand in df.columns:
            df = df.rename(columns={cand: "label"})
            break
if "label" not in df.columns:
    raise ValueError("Rename target column to 'label'.")

def to01(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, np.integer)) and v in (0, 1):
        return int(v)
    s = str(v).strip().lower()
    if s in {"benign", "good", "legit", "legitimate", "safe", "0"}:
        return 0
    if s in {"malicious", "phishing", "bad", "malware", "1"}:
        return 1
    raise ValueError(f"Unknown label value: {v}")

def canonicalize_url(u):
    """
    Canonical form used only to prevent leakage:
    - trims whitespace
    - lowercases scheme + host
    - drops fragment (#...) because it never reaches the server
    - preserves path case
    - preserves query
    - also handles URLs that may be missing a scheme
    """
    s = str(u).strip()
    try:
        p = urlsplit(s)
        has_explicit_scheme_or_netloc = bool(p.scheme or p.netloc)
        if not has_explicit_scheme_or_netloc:
            p = urlsplit("http://" + s)

        scheme = (p.scheme or "").lower()
        netloc = (p.netloc or "").lower()
        path = p.path or ""
        query = p.query or ""

        if has_explicit_scheme_or_netloc:
            return urlunsplit((scheme, netloc, path, query, ""))
        else:
            return netloc + path + (f"?{query}" if query else "")
    except Exception:
        return s

df = df.dropna(subset=["url", "label"]).copy()
df["label"] = df["label"].apply(to01).astype(int)
df["url_raw"] = df["url"].astype(str).str.strip()
df["url_key"] = df["url_raw"].map(canonicalize_url)

conflict_counts = df.groupby("url_key")["label"].nunique()
n_conflicts = int((conflict_counts > 1).sum())
if n_conflicts > 0:
    bad_examples = conflict_counts[conflict_counts > 1].index[:10].tolist()
    raise ValueError(
        f"Found {n_conflicts} canonical URLs with conflicting labels. "
        f"Examples: {bad_examples}"
    )

before = len(df)
df = df.drop_duplicates(subset=["url_key"]).reset_index(drop=True)

df["url"] = df["url_key"]

print("rows after normalized dedup:", len(df), "/", before)
print("normalized duplicates removed:", before - len(df))
print("class counts:", np.bincount(df["label"]))
df.head(3)


In [ ]:
#Create leakage controlled splits by grouping URLs with the same normalised base domain.

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
from urllib.parse import urlsplit

def get_host_key(u):
    s = str(u).strip()
    try:
        p = urlsplit(s)
        if not (p.scheme or p.netloc):
            p = urlsplit("http://" + s)
        return (p.hostname or "").lower()
    except Exception:
        return ""

def get_base_domain_key(u):
    host = get_host_key(u)
    e = extractor(host if host else str(u))
    if e.suffix:
        return f"{e.domain}.{e.suffix}".lower()
    return e.domain.lower()

df["host_key"] = df["url"].map(get_host_key)
df["base_domain"] = df["url"].map(get_base_domain_key)

# First split: 70% train, 30% temp, grouped by normalized base domain
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
train_idx, temp_idx = next(gss1.split(df, groups=df["base_domain"]))

df_train = df.iloc[train_idx].copy()
df_temp = df.iloc[temp_idx].copy()

# Second split: split temp into 15% val / 15% test, grouped by normalized base domain
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp["base_domain"]))

df_val = df_temp.iloc[val_idx].copy().reset_index(drop=True)
df_test = df_temp.iloc[test_idx].copy().reset_index(drop=True)
df_train = df_train.reset_index(drop=True)

for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    counts = np.bincount(part["label"])
    print(f"{name:5s} -> shape={part.shape}, class_counts={counts}")

# Strict overlap checks on the actual grouping key
train_domains = set(df_train["base_domain"])
val_domains = set(df_val["base_domain"])
test_domains = set(df_test["base_domain"])

print("\nBase-domain overlap checks:")
print("Train vs Val :", len(train_domains & val_domains))
print("Train vs Test:", len(train_domains & test_domains))
print("Val vs Test  :", len(val_domains & test_domains))


In [ ]:
# Run overlap checks to confirm that the grouped split reduces duplicate and domain family leakage.

from urllib.parse import urlsplit, urlunsplit

def strip_query_fragment(u):
    try:
        p = urlsplit(str(u))
        scheme = (p.scheme or "").lower()
        netloc = (p.netloc or "").lower()
        path = (p.path or "").rstrip("/")
        return urlunsplit((scheme, netloc, path, "", ""))
    except Exception:
        return str(u).strip()

def show_overlap(a, b, label):
    a, b = set(a), set(b)
    inter = a & b
    print(f"{label}: {len(inter)} Overlaps")
    if len(inter) > 0:
        print("Examples:", list(sorted(inter))[:10])

print("=== Exact canonical URL overlap ===")
show_overlap(df_train["url"], df_val["url"], "Train vs Val")
show_overlap(df_train["url"], df_test["url"], "Train vs Test")
show_overlap(df_val["url"], df_test["url"], "Val vs Test")

print("\n=== Base-domain overlap ===")
show_overlap(df_train["base_domain"], df_val["base_domain"], "Train vs Val")
show_overlap(df_train["base_domain"], df_test["base_domain"], "Train vs Test")
show_overlap(df_val["base_domain"], df_test["base_domain"], "Val vs Test")

print("\n=== Host overlap ===")
show_overlap(df_train["host_key"], df_val["host_key"], "Train vs Val")
show_overlap(df_train["host_key"], df_test["host_key"], "Train vs Test")
show_overlap(df_val["host_key"], df_test["host_key"], "Val vs Test")

print("\n=== Overlap after stripping query/fragment ===")
show_overlap(df_train["url"].map(strip_query_fragment), df_val["url"].map(strip_query_fragment), "Train vs Val")
show_overlap(df_train["url"].map(strip_query_fragment), df_test["url"].map(strip_query_fragment), "Train vs Test")
show_overlap(df_val["url"].map(strip_query_fragment), df_test["url"].map(strip_query_fragment), "Val vs Test")


In [ ]:
# 19 handcrafted lexical URL features used by the lexical and hybrid

IPV4 = re.compile(r"^(?:\d{1,3}\.){3}\d{1,3}$")
SPECIALS = list("./-_@?=%&")

def shannon_entropy(s):
    if not s:
        return 0.0
    c = Counter(s)
    n = float(len(s))
    return -sum((v / n) * math.log2(v / n) for v in c.values())

def lexical_features(url):
    try:
        p = urllib.parse.urlsplit(url)
    except Exception:
        p = urllib.parse.urlsplit("")

    scheme = p.scheme or ""
    host = p.hostname or ""
    path = p.path or ""
    query = p.query or ""

    t = extractor(url)
    sub_count = len([x for x in (t.subdomain or "").split(".") if x])

    feats = {
        "len": len(url),
        "https_flag": 1 if scheme.lower() == "https" else 0,
        "ip_in_host": 1 if IPV4.match(host) else 0,
        "subdomain_count": sub_count,
        "digit_count": sum(ch.isdigit() for ch in url),
        "alpha_count": sum(ch.isalpha() for ch in url),
        "entropy": shannon_entropy(url),
        "path_depth": path.count("/"),
        "query_params": query.count("&") + (1 if query else 0),
    }

    for ch in SPECIALS:
        feats[f"count_{ch}"] = url.count(ch)

    feats["digit_alpha_ratio"] = feats["digit_count"] / (feats["alpha_count"] + 1e-6)
    return feats

FEATURE_ORDER = [
    "len", "https_flag", "ip_in_host", "subdomain_count", "digit_count", "alpha_count",
    "entropy", "path_depth", "query_params",
    "count_.", "count_/", "count_-", "count__", "count_@", "count_?", "count_=", "count_%", "count_&",
    "digit_alpha_ratio"
]

def build_lex_matrix(url_series):
    rows = []
    for u in url_series:
        f = lexical_features(u)
        rows.append([f[k] for k in FEATURE_ORDER])
    return np.asarray(rows, dtype=np.float32)


In [ ]:
# Load DistilBERT tokenizer, create sampled modelling splits, build lexical matrices, and fit the scaler on training data only

tok = AutoTokenizer.from_pretrained("distilbert-base-cased")

def sample_frame(frame, n):
    n = min(n, len(frame))
    return frame.sample(n=n, random_state=SEED).reset_index(drop=True)

if CONFIG["use_sample"]:
    df_train_model = sample_frame(df_train, CONFIG["sample_train"])
    df_val_model   = sample_frame(df_val,   CONFIG["sample_val"])
    df_test_model  = sample_frame(df_test,  CONFIG["sample_test"])
else:
    df_train_model = df_train.reset_index(drop=True)
    df_val_model   = df_val.reset_index(drop=True)
    df_test_model  = df_test.reset_index(drop=True)

print(df_train_model.shape, df_val_model.shape, df_test_model.shape)

X_train_raw = build_lex_matrix(df_train_model["url"])
X_val_raw   = build_lex_matrix(df_val_model["url"])
X_test_raw  = build_lex_matrix(df_test_model["url"])

scaler = StandardScaler().fit(X_train_raw)

X_train = scaler.transform(X_train_raw).astype(np.float32)
X_val   = scaler.transform(X_val_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

y_train = df_train_model["label"].to_numpy()
y_val   = df_val_model["label"].to_numpy()
y_test  = df_test_model["label"].to_numpy()

print(X_train.shape, X_val.shape, X_test.shape)


In [ ]:
# Sanity Check — Tokenizer behavior
sample_urls = [
    "https://PayPal-login-Verify.com/account/update?user=ABC123",
    "http://secure-bank-login.example.com/Verify-Now"
]

for u in sample_urls:
    toks = tok.tokenize(u)
    print("\nURL:", u)
    print("Tokens:", toks[:40])


In [ ]:
# Sanity check: confirm that StandardScaler was fitted on training features only

import pandas as pd
import numpy as np

train_means = np.mean(X_train, axis=0)
train_stds = np.std(X_train, axis=0)

print("Train feature means (first 10):")
print(np.round(train_means[:10], 4))

print("\nTrain feature stds (first 10):")
print(np.round(train_stds[:10], 4))

print("\nVal means should NOT all be exactly zero:")
print(np.round(np.mean(X_val, axis=0)[:10], 4))

In [ ]:
# Create PyTorch Dataset, collate function, and DataLoaders

class URLDataset(Dataset):
    def __init__(self, frame, lex_array):
        self.urls = frame["url"].tolist()
        self.labels = frame["label"].to_numpy(dtype=np.float32)
        self.lex = torch.from_numpy(lex_array)

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        return self.urls[idx], self.lex[idx], self.labels[idx]

def collate_fn(batch):
    urls, lex_feats, labels = zip(*batch)

    enc = tok(
        list(urls),
        padding="max_length",
        truncation=True,
        max_length=CONFIG["max_len"],
        return_tensors="pt"
    )

    lex = torch.stack(lex_feats)
    y = torch.tensor(labels, dtype=torch.float32)

    return enc["input_ids"], enc["attention_mask"], lex, y

LOADER_KWARGS = dict(
    batch_size=CONFIG["batch_size"],
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

train_loader = DataLoader(
    URLDataset(df_train_model, X_train),
    shuffle=True,
    **LOADER_KWARGS
)

val_loader = DataLoader(
    URLDataset(df_val_model, X_val),
    shuffle=False,
    **LOADER_KWARGS
)

test_loader = DataLoader(
    URLDataset(df_test_model, X_test),
    shuffle=False,
    **LOADER_KWARGS
)

len(train_loader), len(val_loader), len(test_loader)


In [ ]:
#  Evaluation metrics and weighted binary cross-entropy loss

def eval_pred(y_true, y_prob, thr=None):
    if thr is None: thr = CONFIG["threshold"]
    y_pred = (y_prob >= thr).astype(int)
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    auc_roc = roc_auc_score(y_true, y_prob)
    return {"acc":float(acc),"prec":float(p),"rec":float(r),"f1":float(f1),"roc_auc":float(auc_roc)}

def make_weighted_loss(train_frame):
    pos = train_frame["label"].sum()
    neg = len(train_frame) - pos
    pos_weight = torch.tensor(neg / max(pos,1), dtype=torch.float32, device=DEVICE)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

criterion = make_weighted_loss(df_train_model)


In [ ]:
# Stronger sanity check — independent random labels
from sklearn.linear_model import LogisticRegression
import numpy as np

rng = np.random.default_rng(42)

p = y_train.mean()
y_train_random = rng.binomial(1, p, size=len(y_train))

print("Agreement with true training labels:", (y_train_random == y_train).mean())

clf = LogisticRegression(max_iter=300, class_weight="balanced", solver="lbfgs")
clf.fit(X_train, y_train_random)

test_prob = clf.predict_proba(X_test)[:, 1]
print("Independent-random-label lexical baseline on TEST:")
print(eval_pred(y_test, test_prob))


In [ ]:
# Define the three model architectures

class LexicalMLPBaseline(nn.Module):
    def __init__(self, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(19, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )

    def forward(self, lex_feats):
        return self.net(lex_feats).squeeze(1)


class DistilBertOnlyClassifier(nn.Module):
    def __init__(self, pooling="mean", dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained("distilbert-base-cased")
        self.pooling = pooling
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(self.bert.config.dim, 1)

    def pool(self, last_hidden_state, attention_mask):
        if self.pooling == "cls":
            return last_hidden_state[:, 0]
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return summed / denom

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.drop(self.pool(out.last_hidden_state, attention_mask))
        return self.head(pooled).squeeze(1)


class DistilBertLexHybrid(nn.Module):
    def __init__(self, pooling="mean", dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained("distilbert-base-cased")
        self.pooling = pooling

        self.lex_mlp = nn.Sequential(
            nn.Linear(19, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.ReLU()
        )

        self.drop = nn.Dropout(dropout)

        self.fuse = nn.Sequential(
            nn.Linear(self.bert.config.dim + 32, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def pool(self, last_hidden_state, attention_mask):
        if self.pooling == "cls":
            return last_hidden_state[:, 0]
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return summed / denom

    def forward(self, input_ids, attention_mask, lex_feats):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.drop(self.pool(out.last_hidden_state, attention_mask))
        lex_emb = self.lex_mlp(lex_feats)
        fused = torch.cat([pooled, lex_emb], dim=1)
        return self.fuse(fused).squeeze(1)

In [ ]:
# Shared optimiser, epoch-running, freezing, and early-stopping utilities

def freeze_bert(model, freeze=True):
    if hasattr(model, "bert"):
        for p in model.bert.parameters():
            p.requires_grad = not freeze

def make_optimizer(model):
    if hasattr(model, "bert"):
        bert_params, head_params = [], []
        for name, p in model.named_parameters():
            if name.startswith("bert."): bert_params.append(p)
            else: head_params.append(p)
        return torch.optim.AdamW([
            {"params": bert_params, "lr": CONFIG["lr_bert"], "weight_decay": CONFIG["weight_decay"]},
            {"params": head_params, "lr": CONFIG["lr_head"], "weight_decay": CONFIG["weight_decay"]},
        ])
    return torch.optim.AdamW(model.parameters(), lr=CONFIG["lr_head"], weight_decay=CONFIG["weight_decay"])

def run_epoch_lex(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(mode=train)
    all_probs, all_labels, running_loss = [], [], 0.0
    for input_ids, attention_mask, lex_feats, y in tqdm(loader, desc=("train" if train else "eval"), leave=False):
        lex_feats = lex_feats.to(DEVICE); y = y.to(DEVICE)
        logits = model(lex_feats); loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        running_loss += loss.item() * y.size(0)
        all_probs.append(torch.sigmoid(logits).detach().cpu().numpy()); all_labels.append(y.detach().cpu().numpy())
    y_true = np.concatenate(all_labels); y_prob = np.concatenate(all_probs)
    m = eval_pred(y_true, y_prob); m["loss"] = float(running_loss / len(loader.dataset)); return m

def run_epoch_bert(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(mode=train)
    all_probs, all_labels, running_loss = [], [], 0.0
    for input_ids, attention_mask, lex_feats, y in tqdm(loader, desc=("train" if train else "eval"), leave=False):
        input_ids = input_ids.to(DEVICE); attention_mask = attention_mask.to(DEVICE); y = y.to(DEVICE)
        logits = model(input_ids, attention_mask); loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        running_loss += loss.item() * y.size(0)
        all_probs.append(torch.sigmoid(logits).detach().cpu().numpy()); all_labels.append(y.detach().cpu().numpy())
    y_true = np.concatenate(all_labels); y_prob = np.concatenate(all_probs)
    m = eval_pred(y_true, y_prob); m["loss"] = float(running_loss / len(loader.dataset)); return m

def run_epoch_hybrid(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(mode=train)
    all_probs, all_labels, running_loss = [], [], 0.0
    for input_ids, attention_mask, lex_feats, y in tqdm(loader, desc=("train" if train else "eval"), leave=False):
        input_ids = input_ids.to(DEVICE); attention_mask = attention_mask.to(DEVICE); lex_feats = lex_feats.to(DEVICE); y = y.to(DEVICE)
        logits = model(input_ids, attention_mask, lex_feats); loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        running_loss += loss.item() * y.size(0)
        all_probs.append(torch.sigmoid(logits).detach().cpu().numpy()); all_labels.append(y.detach().cpu().numpy())
    y_true = np.concatenate(all_labels); y_prob = np.concatenate(all_probs)
    m = eval_pred(y_true, y_prob); m["loss"] = float(running_loss / len(loader.dataset)); return m

def train_with_early_stopping(model, run_epoch_fn, freeze_bert_first=False):
    model = model.to(DEVICE)
    optimizer = make_optimizer(model)
    if freeze_bert_first: freeze_bert(model, True)
    best_val, best_state, no_improve = None, None, 0
    for epoch in range(1, CONFIG["epochs"] + 1):
        if freeze_bert_first and epoch == (CONFIG["freeze_bert_epochs"] + 1):
            freeze_bert(model, False)
        tr = run_epoch_fn(model, train_loader, optimizer=optimizer)
        va = run_epoch_fn(model, val_loader, optimizer=None)
        print(f"Epoch {epoch}: train loss {tr['loss']:.4f}, f1 {tr['f1']:.4f} | val loss {va['loss']:.4f}, f1 {va['f1']:.4f}, auc {va['roc_auc']:.4f}")
        if (best_val is None) or (va["f1"] > best_val["f1"]):
            best_val = va; best_state = {k:v.cpu() for k,v in model.state_dict().items()}; no_improve = 0
        else:
            no_improve += 1
            if no_improve >= CONFIG["patience"]:
                print("Early stopping."); break
    if best_state is not None:
        model.load_state_dict(best_state); model.to(DEVICE)
    return model, best_val


In [ ]:
# Train and evaluate the lexical only MLP baseline

lex_model = LexicalMLPBaseline(dropout=CONFIG["dropout"]).to(DEVICE)
lex_model, best_val_lex = train_with_early_stopping(lex_model, run_epoch_lex, freeze_bert_first=False)
lex_test_metrics = run_epoch_lex(lex_model, test_loader, optimizer=None)
print("TEST (lexical MLP):", lex_test_metrics)


In [ ]:
# Train and evaluate the DistilBERT only baseline

bert_results = {}
for pooling in CONFIG["pooling_options"]:
    print("\n=== Training DistilBERT-only with pooling =", pooling, "===")
    model = DistilBertOnlyClassifier(pooling=pooling, dropout=CONFIG["dropout"]).to(DEVICE)
    model, best_val = train_with_early_stopping(model, run_epoch_bert, freeze_bert_first=True)
    test_metrics = run_epoch_bert(model, test_loader, optimizer=None)
    bert_results[pooling] = {"model": model, "best_val": best_val, "test": test_metrics}
    print("TEST:", test_metrics)

best_pooling = max(CONFIG["pooling_options"], key=lambda p: bert_results[p]["best_val"]["f1"])
best_bert_model = bert_results[best_pooling]["model"]
best_bert_test_metrics = bert_results[best_pooling]["test"]
print("Best pooling:", best_pooling)
print("Best DistilBERT-only test metrics:", best_bert_test_metrics)


In [ ]:
# Set the pooling strategy used by the final hybrid model
best_pooling = "mean"

In [ ]:
# Train and evaluate the final hybrid model

hybrid_model = DistilBertLexHybrid(pooling=best_pooling, dropout=CONFIG["dropout"]).to(DEVICE)
hybrid_model, best_val_hybrid = train_with_early_stopping(hybrid_model, run_epoch_hybrid, freeze_bert_first=True)
hybrid_test_metrics = run_epoch_hybrid(hybrid_model, test_loader, optimizer=None)
print("TEST (hybrid):", hybrid_test_metrics)


In [ ]:
# Define helper functions for validation-based threshold tuning

def find_best_threshold(y_true, y_prob, grid=np.linspace(0.05, 0.95, 181)):
    best = None
    for thr in grid:
        metrics = eval_pred(y_true, y_prob, thr=thr)
        cand = {"threshold": float(thr), **metrics}
        if best is None or cand["f1"] > best["f1"]:
            best = cand
    return best

def collect_probs_lex(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for input_ids, attention_mask, lex_feats, y in loader:
        lex_feats = lex_feats.to(DEVICE)
        with torch.no_grad():
            logits = model(lex_feats)
            probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs), None

def collect_probs_bert(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for input_ids, attention_mask, lex_feats, y in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        with torch.no_grad():
            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs), None

def collect_probs_hybrid(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for input_ids, attention_mask, lex_feats, y in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        lex_feats = lex_feats.to(DEVICE)
        with torch.no_grad():
            logits = model(input_ids, attention_mask, lex_feats)
            probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs), None


In [ ]:
#Run threshold tuning for each model and evaluate tuned thresholds on the test set

# Lexical threshold tuning
y_val_lex, p_val_lex, _ = collect_probs_lex(lex_model, val_loader)
best_thr_lex = find_best_threshold(y_val_lex, p_val_lex)

y_test_lex, p_test_lex, _ = collect_probs_lex(lex_model, test_loader)
lex_test_metrics_tuned = eval_pred(y_test_lex, p_test_lex, thr=best_thr_lex["threshold"])

print("Best lexical validation threshold:", best_thr_lex)
print("TEST (lexical MLP, tuned threshold):", lex_test_metrics_tuned)


# DistilBERT only threshold tuning
y_val_bert, p_val_bert, _ = collect_probs_bert(best_bert_model, val_loader)
best_thr_bert = find_best_threshold(y_val_bert, p_val_bert)

y_test_bert, p_test_bert, _ = collect_probs_bert(best_bert_model, test_loader)
bert_test_metrics_tuned = eval_pred(y_test_bert, p_test_bert, thr=best_thr_bert["threshold"])

print("Best DistilBERT validation threshold:", best_thr_bert)
print("TEST (DistilBERT-only, tuned threshold):", bert_test_metrics_tuned)


# Hybrid threshold tuning
y_val_hybrid, p_val_hybrid, _ = collect_probs_hybrid(hybrid_model, val_loader)
best_thr_hybrid = find_best_threshold(y_val_hybrid, p_val_hybrid)

y_test_hybrid, p_test_hybrid, _ = collect_probs_hybrid(hybrid_model, test_loader)
hybrid_test_metrics_tuned = eval_pred(
    y_test_hybrid,
    p_test_hybrid,
    thr=best_thr_hybrid["threshold"]
)

print("Best hybrid validation threshold:", best_thr_hybrid)
print("TEST (hybrid, tuned threshold):", hybrid_test_metrics_tuned)
